### TOPIC : NLP
### MODEL : AMAZON REVIEW

## PART 1 — Original Score Labels (1 to 5)

In [1]:
# ── Imports ──────────────────────────────────────────────────────────────────
import warnings
warnings.filterwarnings('ignore')

import pandas as pd
import numpy as np
import string

# FIX: download stopwords if not already present
import nltk
try:
    from nltk.corpus import stopwords
    _ = stopwords.words('english')          # trigger load
except LookupError:
    nltk.download('stopwords')
    from nltk.corpus import stopwords

from sklearn.feature_extraction.text import CountVectorizer
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import confusion_matrix, accuracy_score

pd.set_option('display.max_columns', None)
print('All imports OK')

All imports OK


### Load Data

In [2]:
# FIX: added error_bad_lines=False → on_bad_lines='skip' (pandas ≥ 1.3)
Amazon = pd.read_csv('Reviews.csv', encoding='utf-8', on_bad_lines='skip')
print('Shape:', Amazon.shape)
Amazon.head(3)

Shape: (568454, 10)


,Id,ProductId,UserId,ProfileName,HelpfulnessNumerator,HelpfulnessDenominator,Score,Time,Summary,Text
0,1,B001E4KFG0,A3SGXH7AUHU8GW,delmartian,1,1,5,1303862400,Good Quality Dog Food,I have bought several of the Vitality canned d...
1,2,B00813GRG4,A1D87F6ZCVE5NK,dll pa,0,0,1,1346976000,Not as Advertised,Product arrived labeled as Jumbo Salted Peanut...
2,3,B000LQOCH0,ABXLMWJIXXAIN,"Natalia Corres ""Natalia Corres""",1,1,4,1219017600,"""Delight"" says it all",This is a confection that has been around a fe...


### Data Cleaning

In [3]:
# Keep only required columns
Amazon = Amazon[['Score', 'Text']].copy()

# FIX: drop rows where Score is NaN (cannot be a target label)
Amazon.dropna(subset=['Score'], inplace=True)

# Fill missing text with empty string
Amazon['Text'] = Amazon['Text'].fillna('').astype(str)

# Lowercase
Amazon['Text'] = Amazon['Text'].str.lower()

# FIX: convert Score to int (it may be read as float)
Amazon['Score'] = Amazon['Score'].astype(int)

print('Null counts after cleaning:')
print(Amazon.isnull().sum())
print('\nScore distribution:')
print(Amazon['Score'].value_counts().sort_index())

Null counts after cleaning:
Score    0
Text     0
dtype: int64

Score distribution:
Score
1     52268
2     29769
3     42640
4     80655
5    363122
Name: count, dtype: int64


### Stop-Words Removal & Text Processing

In [4]:
Stop_Words = set(stopwords.words('english'))   # FIX: use set for O(1) lookup

def text_process(message):
    """Remove punctuation and stop words, return list of tokens."""
    # FIX: message must be a str; coerce just in case
    message = str(message)
    nopunc = ''.join(ch for ch in message if ch not in string.punctuation)
    return [word for word in nopunc.split() if word not in Stop_Words]

# Quick sanity check
print(text_process('This is a great product! Loved it.'))

['This', 'great', 'product', 'Loved']


### Count Vectorization

In [5]:
# FIX: CountVectorizer with a custom analyzer; fit on the Text column
Text_Count = CountVectorizer(analyzer=text_process)
Text_Count.fit(Amazon['Text'])

Amazon_X = Text_Count.transform(Amazon['Text'])
print('Feature matrix shape:', Amazon_X.shape)

Feature matrix shape: (568454, 240626)


### Train / Test Split

In [6]:
# FIX: added random_state for reproducibility
X_Train, X_Test, Y_Train, Y_Test = train_test_split(
    Amazon_X, Amazon['Score'],
    test_size=0.2,
    random_state=42
)

print('Shapes')
print('X_Train:', X_Train.shape)
print('Y_Train:', Y_Train.shape)
print('X_Test :', X_Test.shape)
print('Y_Test :', Y_Test.shape)

Shapes
X_Train: (454763, 240626)
Y_Train: (454763,)
X_Test : (113691, 240626)
Y_Test : (113691,)


### Decision Tree Model

In [ ]:
DT = DecisionTreeClassifier(random_state=42)   # FIX: random_state for reproducibility
DT.fit(X_Train, Y_Train)

Predicted_DT = DT.predict(X_Test)

Confusion_DT = confusion_matrix(Y_Test, Predicted_DT)
print('Confusion Matrix — Decision Tree:')
print(Confusion_DT)

In [ ]:
DT_Accuracy = accuracy_score(Y_Test, Predicted_DT) * 100
print(f'Decision Tree Accuracy: {DT_Accuracy:.2f}%')

### Naïve Bayes Model

In [ ]:
# FIX: MultinomialNB requires non-negative features — CountVectorizer output is fine
NB = MultinomialNB()
NB.fit(X_Train, Y_Train)

Predicted_NB = NB.predict(X_Test)

Confusion_NB = confusion_matrix(Y_Test, Predicted_NB)
print('Confusion Matrix — Naïve Bayes:')
print(Confusion_NB)

In [ ]:
NB_Accuracy = accuracy_score(Y_Test, Predicted_NB) * 100
print(f'Naïve Bayes Accuracy: {NB_Accuracy:.2f}%')

---
## PART 2 — Converted / Remapped Score Labels

| Original Score | New Label | Meaning |
|:-:|:-:|:-:|
| 1 | 1 | Negative |
| 2 | 1 | Negative |
| 3 | 2 | Neutral  |
| 4 | 3 | Positive |
| 5 | 3 | Positive |


In [ ]:
# Reload fresh copy so Part 1 is unaffected
Amazon2 = pd.read_csv('Reviews.csv', encoding='utf-8', on_bad_lines='skip')
Amazon2 = Amazon2[['Score', 'Text']].copy()
Amazon2.dropna(subset=['Score'], inplace=True)
Amazon2['Text'] = Amazon2['Text'].fillna('').astype(str).str.lower()
Amazon2['Score'] = Amazon2['Score'].astype(int)

# FIX: mapping must include ALL original values; original code missed Score==1
# Corrected mapping: 1→1 (neg), 2→1 (neg), 3→2 (neutral), 4→3 (pos), 5→3 (pos)
Amazon2['Score'] = Amazon2['Score'].replace({1: 1, 2: 1, 3: 2, 4: 3, 5: 3})

print('Remapped Score distribution:')
print(Amazon2['Score'].value_counts().sort_index())

### Stop-Words Removal & Text Processing (Part 2)

In [ ]:
# text_process function is the same — already defined above
Amazon2['Text'].apply(text_process).head(3)

### Count Vectorization (Part 2)

In [ ]:
Text_Count2 = CountVectorizer(analyzer=text_process)
Text_Count2.fit(Amazon2['Text'])

Amazon2_X = Text_Count2.transform(Amazon2['Text'])
print('Feature matrix shape:', Amazon2_X.shape)

### Train / Test Split (Part 2)

In [ ]:
X_Train2, X_Test2, Y_Train2, Y_Test2 = train_test_split(
    Amazon2_X, Amazon2['Score'],
    test_size=0.2,
    random_state=42
)

print('Shapes')
print('X_Train2:', X_Train2.shape)
print('Y_Train2:', Y_Train2.shape)
print('X_Test2 :', X_Test2.shape)
print('Y_Test2 :', Y_Test2.shape)

### Decision Tree Model (Part 2)

In [ ]:
DT2 = DecisionTreeClassifier(random_state=42)
DT2.fit(X_Train2, Y_Train2)

Predicted_DT2 = DT2.predict(X_Test2)

Confusion_DT2 = confusion_matrix(Y_Test2, Predicted_DT2)
print('Confusion Matrix — Decision Tree (Part 2):')
print(Confusion_DT2)

In [ ]:
DT2_Accuracy = accuracy_score(Y_Test2, Predicted_DT2) * 100
print(f'Decision Tree Accuracy (Part 2): {DT2_Accuracy:.2f}%')

### Naïve Bayes Model (Part 2)

In [ ]:
NB2 = MultinomialNB()
NB2.fit(X_Train2, Y_Train2)

Predicted_NB2 = NB2.predict(X_Test2)

Confusion_NB2 = confusion_matrix(Y_Test2, Predicted_NB2)
print('Confusion Matrix — Naïve Bayes (Part 2):')
print(Confusion_NB2)

In [ ]:
NB2_Accuracy = accuracy_score(Y_Test2, Predicted_NB2) * 100
print(f'Naïve Bayes Accuracy (Part 2): {NB2_Accuracy:.2f}%')

---
## Summary

| Part | Model | Accuracy |
|:--|:--|:--|
| Part 1 (5-class) | Decision Tree | `DT_Accuracy` % |
| Part 1 (5-class) | Naïve Bayes   | `NB_Accuracy` % |
| Part 2 (3-class) | Decision Tree | `DT2_Accuracy` % |
| Part 2 (3-class) | Naïve Bayes   | `NB2_Accuracy` % |

In [ ]:
summary = pd.DataFrame({
    'Part'   : ['Part 1 (5-class)', 'Part 1 (5-class)', 'Part 2 (3-class)', 'Part 2 (3-class)'],
    'Model'  : ['Decision Tree', 'Naïve Bayes', 'Decision Tree', 'Naïve Bayes'],
    'Accuracy (%)': [round(DT_Accuracy, 2), round(NB_Accuracy, 2),
                     round(DT2_Accuracy, 2), round(NB2_Accuracy, 2)]
})
print(summary.to_string(index=False))